# IE4483 Mini Project - Dogs vs Cats (ResNet-18)
Run the cells in order on a GPU runtime. The dataset must be in `/content/datasets` (folders `train/`, `val/`, `test/`), or as `/content/datasets.zip`, which is then unzipped.

## Setup

In [ ]:
import os, random, time, copy, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models

BASE_DIR = '/content'
ZIP_PATH = f'{BASE_DIR}/datasets.zip'
DATA_ROOT = Path(BASE_DIR) / 'datasets'          # contains train/, val/, test/
SUBMISSION_PATH = Path(BASE_DIR) / 'submission.csv'

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
MEAN = [0.485, 0.456, 0.406]                     # ImageNet statistics (backbone pretrained on ImageNet)
STD = [0.229, 0.224, 0.225]

N_TRIALS = 8                  # random search
TUNE_EPOCHS = 3
TUNE_TRAIN_PER_CLASS = 1000   # subset for tuning and ablations
TUNE_VAL_PER_CLASS = 250
FINAL_EPOCHS = 6              # final training
FINAL_PATIENCE = 3
REPRO_SEEDS = [0, 1, 2]       # reproducibility check
REPRO_EPOCHS = 3
REPRO_TRAIN_PER_CLASS = None  # None = all training images
ABL_EPOCHS = 3                # ablations
ABL_LR = 1e-3

EXPECTED_COUNTS = {"train_cat": 10000, "train_dog": 10000,
                   "val_cat": 2500, "val_dog": 2500, "test": 500}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
print("Device:", device)

## 1. Data

In [ ]:
if not DATA_ROOT.exists():
    assert os.path.exists(ZIP_PATH), f"Put datasets.zip in {BASE_DIR} first"
    assert zipfile.is_zipfile(ZIP_PATH), "datasets.zip is incomplete: upload it again"
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(BASE_DIR)

IMG_EXT = ('.jpg', '.jpeg', '.png')

def is_image(p):
    name = Path(p).name
    return name.lower().endswith(IMG_EXT) and not name.startswith('._')     # skips macOS junk files

def count_images(folder):
    return sum(1 for f in Path(folder).iterdir() if is_image(f))

def verify_dataset_structure(data_root=DATA_ROOT, expected=None):
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"Dataset directory '{data_root}' not found.")
    counts = {
        "train_cat": count_images(data_root / "train" / "cat"),
        "train_dog": count_images(data_root / "train" / "dog"),
        "val_cat": count_images(data_root / "val" / "cat"),
        "val_dog": count_images(data_root / "val" / "dog"),
        "test": count_images(data_root / "test"),
    }
    assert all(v > 0 for v in counts.values()), f"Empty folder found: {counts}"
    print(counts)
    if expected is not None and counts != expected:
        print(f"WARNING: counts differ from the assignment {expected}")
    return counts

DATASET_COUNTS = verify_dataset_structure(DATA_ROOT, EXPECTED_COUNTS)

In [ ]:
def get_transforms(img_size=IMG_SIZE, augment=True):
    # Augmentation on the training set only; validation and test stay deterministic
    if augment:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, eval_tf

train_tf, eval_tf = get_transforms()

In [ ]:
class UnlabelledImageDataset(Dataset):
    # Test set: flat folder without labels, returns (image, file name)
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_files = sorted(f for f in self.root_dir.glob("*") if is_image(f))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_path.name


def balanced_subset(dataset, n_per_class, seed=SEED):
    if n_per_class is None:
        return dataset
    rng = np.random.RandomState(seed)
    targets = np.array(dataset.targets)
    indices = []
    for c in np.unique(targets):
        idx = np.where(targets == c)[0]
        indices.extend(rng.choice(idx, size=min(n_per_class, len(idx)), replace=False).tolist())
    rng.shuffle(indices)
    return Subset(dataset, [int(i) for i in indices])


def make_loaders(n_train_per_class=None, n_val_per_class=None, batch_size=BATCH_SIZE,
                 img_size=IMG_SIZE, augment=True, seed=SEED):
    # Returns (train_loader, val_loader, test_loader). Labels: cat=0, dog=1 (alphabetical folder order)
    train_tf, eval_tf = get_transforms(img_size=img_size, augment=augment)
    train_full = datasets.ImageFolder(str(DATA_ROOT / "train"), transform=train_tf, is_valid_file=is_image)
    val_full = datasets.ImageFolder(str(DATA_ROOT / "val"), transform=eval_tf, is_valid_file=is_image)
    assert train_full.class_to_idx == {"cat": 0, "dog": 1}, "Class mapping mismatch!"

    train_ds = balanced_subset(train_full, n_train_per_class, seed)
    val_ds = balanced_subset(val_full, n_val_per_class, seed)

    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_ds = UnlabelledImageDataset(DATA_ROOT / "test", transform=eval_tf)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


set_seed()
train_loader, val_loader, test_loader = make_loaders()
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

In [ ]:
# Original image and random augmentations (one cat, one dog)
def show_augmentation_examples(n_aug=5):
    raw = datasets.ImageFolder(str(DATA_ROOT / "train"), is_valid_file=is_image)
    aug_tf, _ = get_transforms(IMG_SIZE, augment=True)
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    fig, axes = plt.subplots(2, n_aug + 1, figsize=(3 * (n_aug + 1), 6.4))
    for r, idx in enumerate([0, len(raw) - 1]):
        img, lab = raw[idx]
        axes[r, 0].imshow(img)
        axes[r, 0].set_title("original (" + ("dog" if lab == 1 else "cat") + ")")
        for c in range(1, n_aug + 1):
            t = aug_tf(img)
            axes[r, c].imshow((t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
            axes[r, c].set_title(f"augmented {c}")
        for ax in axes[r]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_augmentation_examples()

## 2. Model

In [ ]:
def build_model(arch="resnet18", pretrained=True, freeze_backbone=False):
    model = getattr(models, arch)(weights="DEFAULT" if pretrained else None)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 2)     # new head: 512 features -> 2 classes
    return model.to(device)


def make_optimizer(model, lr=1e-3, weight_decay=1e-4, backbone_lr_mult=0.1):
    # Backbone learns 10x slower than the new head, to keep the pretrained features
    head_params = list(model.fc.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    groups = [{"params": head_params, "lr": lr}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": lr * backbone_lr_mult})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)


criterion = nn.CrossEntropyLoss()

In [ ]:
# Architecture figure (shapes measured with a dummy image)
from matplotlib.patches import FancyBboxPatch

def draw_architecture():
    m = build_model("resnet18", pretrained=False).eval()
    shapes = {}
    def make_hook(name):
        def hook(module, inp, out):
            shapes[name] = tuple(out.shape[1:])
        return hook
    for name in ["maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool", "fc"]:
        getattr(m, name).register_forward_hook(make_hook(name))
    with torch.no_grad():
        m(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))

    fmt = lambda s: "x".join(str(v) for v in s)
    boxes = [
        ("Input", "RGB image\nresize + normalise", fmt((3, IMG_SIZE, IMG_SIZE)), "#dddddd"),
        ("Stem", "conv 7x7/2 + BN + ReLU\nmaxpool 3x3/2", fmt(shapes["maxpool"]), "#bcd7f0"),
        ("layer1", "2 x BasicBlock (64)", fmt(shapes["layer1"]), "#bcd7f0"),
        ("layer2", "2 x BasicBlock (128)\nstride 2", fmt(shapes["layer2"]), "#bcd7f0"),
        ("layer3", "2 x BasicBlock (256)\nstride 2", fmt(shapes["layer3"]), "#bcd7f0"),
        ("layer4", "2 x BasicBlock (512)\nstride 2", fmt(shapes["layer4"]), "#bcd7f0"),
        ("Global avg pool", "flatten", fmt(shapes["avgpool"]) + " -> 512", "#bcd7f0"),
        ("Linear (new head)", "512 -> 2", fmt(shapes["fc"]), "#f6c9a0"),
        ("Output", "softmax: P(cat), P(dog)\ncross-entropy loss", "2", "#c8e6c9"),
    ]
    fig, ax = plt.subplots(figsize=(19, 3.6))
    ax.set_xlim(0, 2.0 * len(boxes)); ax.set_ylim(0, 3); ax.axis("off")
    for i, (title, sub, shp, color) in enumerate(boxes):
        x0 = 2.0 * i + 0.1
        ax.add_patch(FancyBboxPatch((x0, 0.9), 1.6, 1.5, boxstyle="round,pad=0.03", fc=color, ec="black"))
        ax.text(x0 + 0.8, 2.05, title, ha="center", va="center", fontsize=9, fontweight="bold")
        ax.text(x0 + 0.8, 1.5, sub, ha="center", va="center", fontsize=7)
        ax.text(x0 + 0.8, 0.55, shp, ha="center", va="center", fontsize=8, family="monospace")
        if i < len(boxes) - 1:
            ax.annotate("", xy=(x0 + 1.95, 1.65), xytext=(x0 + 1.65, 1.65), arrowprops=dict(arrowstyle="->"))
    ax.text(0.1, 2.85, "Blue: pretrained backbone (lr x 0.1)   |   Orange: new head   |   shapes: channels x height x width",
            fontsize=8.5, va="center")
    plt.tight_layout()
    plt.show()

draw_architecture()

## 3. Training and hyperparameter search

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                out = model(x)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, lr=1e-3, weight_decay=1e-4, epochs=5, patience=3,
        verbose=True, backbone_lr_mult=0.1):
    # Keeps the weights of the best validation epoch (in memory, nothing is written to disk)
    optimizer = make_optimizer(model, lr, weight_decay, backbone_lr_mult)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state, bad_epochs = -1.0, None, 0

    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_acc = run_epoch(model, val_loader, criterion)
        scheduler.step()
        for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(v)
        if verbose:
            print(f"Epoch {epoch+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time()-t0:.0f}s")

        if va_acc > best_acc:
            best_acc, bad_epochs = va_acc, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_acc


@torch.no_grad()
def predict_probs(model, loader):
    # Returns (probabilities [N, 2], labels for val / file names for test)
    model.eval()
    probs, extras = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), dim=1).cpu())
        extras.extend(y.tolist() if torch.is_tensor(y) else list(y))
    return torch.cat(probs).numpy(), extras


def run_experiment(arch="resnet18", pretrained=True, freeze_backbone=False, augment=True,
                   img_size=IMG_SIZE, lr=1e-3, weight_decay=1e-4, epochs=3,
                   n_train_per_class=TUNE_TRAIN_PER_CLASS, n_val_per_class=TUNE_VAL_PER_CLASS,
                   backbone_lr_mult=0.1, seed=SEED):
    # One complete run with its own seed, loaders and model
    set_seed(seed)
    tr, va, _ = make_loaders(n_train_per_class, n_val_per_class, img_size=img_size, augment=augment, seed=seed)
    m = build_model(arch, pretrained, freeze_backbone)
    t0 = time.time()
    hist, best = fit(m, tr, va, lr=lr, weight_decay=weight_decay, epochs=epochs, patience=epochs,
                     verbose=False, backbone_lr_mult=backbone_lr_mult)
    return {"best_val_acc": best, "train_time_s": time.time() - t0, "history": hist}

In [ ]:
# Random search on a subset; log-scale sampling; selection on the validation set only
rng = np.random.RandomState(0)
rows = []
for t in range(N_TRIALS):
    lr = 10 ** rng.uniform(-4, -2)
    wd = 10 ** rng.uniform(-6, -3)
    freeze = bool(rng.rand() < 0.3)
    r = run_experiment(freeze_backbone=freeze, lr=lr, weight_decay=wd, epochs=TUNE_EPOCHS)
    rows.append({"lr": lr, "weight_decay": wd, "freeze_backbone": freeze, "val_acc": r["best_val_acc"]})
    print(f"Trial {t+1}/{N_TRIALS}: lr={lr:.2e} wd={wd:.2e} freeze={freeze} -> val acc {r['best_val_acc']:.4f}")

results_df = pd.DataFrame(rows).sort_values("val_acc", ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for frozen, marker, label in [(False, "o", "fine-tuned backbone"), (True, "s", "frozen backbone")]:
    sub = results_df[results_df.freeze_backbone == frozen]
    ax.scatter(sub.lr, sub.val_acc, marker=marker, s=60, label=label)
ax.set_xscale("log"); ax.set_xlabel("learning rate (log scale)"); ax.set_ylabel("validation accuracy")
ax.set_title("Random search"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Final training and validation

In [ ]:
best_cfg = results_df.iloc[0]
FINAL_CFG = {"lr": float(best_cfg["lr"]), "weight_decay": float(best_cfg["weight_decay"]),
             "freeze_backbone": bool(best_cfg["freeze_backbone"])}
print(FINAL_CFG)

set_seed()
train_loader, val_loader, test_loader = make_loaders()
final_model = build_model(freeze_backbone=FINAL_CFG["freeze_backbone"])
n_total = sum(p.numel() for p in final_model.parameters())
n_train = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
print(f"Parameters: {n_total:,} total, {n_train:,} trainable")

history, best_val_acc = fit(final_model, train_loader, val_loader, lr=FINAL_CFG["lr"],
                            weight_decay=FINAL_CFG["weight_decay"], epochs=FINAL_EPOCHS,
                            patience=FINAL_PATIENCE)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], marker="o", label="train")
ax[0].plot(history["val_loss"], marker="o", label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], marker="o", label="train")
ax[1].plot(history["val_acc"], marker="o", label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

val_full = datasets.ImageFolder(str(DATA_ROOT / "val"), transform=eval_tf, is_valid_file=is_image)
val_eval_loader = DataLoader(val_full, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

val_probs, val_labels = predict_probs(final_model, val_eval_loader)
val_labels = np.array(val_labels)
val_preds = val_probs.argmax(1)
val_acc = float((val_preds == val_labels).mean())
print(f"Validation accuracy: {val_acc:.4f}  ({int((val_preds != val_labels).sum())} errors out of {len(val_labels)})")
print(classification_report(val_labels, val_preds, target_names=["cat", "dog"], digits=4))

cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["cat", "dog"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["cat", "dog"])
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Validation confusion matrix\n(accuracy {val_acc:.4f})")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

val_df = pd.DataFrame({"path": [s[0] for s in val_full.samples], "true": val_labels,
                       "pred": val_preds, "conf": val_probs.max(1)})

## 5. Reproducibility (several seeds)

In [ ]:
repro_rows = []
for s in REPRO_SEEDS:
    r = run_experiment(freeze_backbone=FINAL_CFG["freeze_backbone"], lr=FINAL_CFG["lr"],
                       weight_decay=FINAL_CFG["weight_decay"], epochs=REPRO_EPOCHS,
                       n_train_per_class=REPRO_TRAIN_PER_CLASS, n_val_per_class=None, seed=s)
    repro_rows.append({"seed": s, "val_acc": r["best_val_acc"]})
    print(f"seed {s}: val acc {r['best_val_acc']:.4f}")

repro_df = pd.DataFrame(repro_rows)
std = repro_df.val_acc.std(ddof=1) if len(repro_df) > 1 else 0.0
print(f"Validation accuracy over {len(repro_df)} seeds: {repro_df.val_acc.mean():.4f} +/- {std:.4f} (mean +/- std)")

## 6. Test predictions and submission.csv

In [ ]:
test_ds = test_loader.dataset
test_probs, test_files = predict_probs(final_model, test_loader)
assert [p.name for p in test_ds.image_files] == list(test_files)
test_preds = test_probs.argmax(1)                       # 1 = dog, 0 = cat

test_df = pd.DataFrame({"file": test_files, "path": [str(p) for p in test_ds.image_files],
                        "pred": test_preds, "label_name": np.where(test_preds == 1, "dog", "cat"),
                        "confidence": test_probs.max(1)})

use_extension, numeric_ids, sample = False, True, None
if SAMPLE_SUBMISSION and os.path.exists(SAMPLE_SUBMISSION):
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    use_extension = str(sample.iloc[0, 0]).lower().endswith(IMG_EXT)
    numeric_ids = pd.api.types.is_numeric_dtype(sample.iloc[:, 0])

ids = list(test_df.file) if use_extension else [Path(f).stem for f in test_df.file]
if numeric_ids and all(str(i).isdigit() for i in ids):
    ids = [int(i) for i in ids]

submission = pd.DataFrame({"id": ids, "label": test_df.pred.values}).sort_values("id").reset_index(drop=True)
if sample is not None:
    submission.columns = list(sample.columns)
    print("Ids identical to the sample:", set(submission.iloc[:, 0]) == set(sample.iloc[:, 0]))
submission.to_csv(SUBMISSION_PATH, index=False)

sub = pd.read_csv(SUBMISSION_PATH)
print("Rows:", len(sub), "| unique ids:", sub.iloc[:, 0].nunique(), "| ids = 1..N:",
      sub.iloc[:, 0].tolist() == list(range(1, len(sub) + 1)))
print("Labels (0=cat, 1=dog):", sub.iloc[:, 1].value_counts().to_dict())
assert len(sub) == len(test_ds) and sub.iloc[:, 0].is_unique and set(sub.iloc[:, 1]) <= {0, 1}
# from google.colab import files; files.download("submission.csv")     # optional: download to your computer

## 7. Error analysis (Grad-CAM)

In [ ]:
def grad_cam(model, x):
    # Returns (heat map in 0..1, predicted class, probabilities) for one normalised image [1, 3, H, W]
    model.eval()
    store = {}
    def fwd_hook(module, inp, out):
        out.retain_grad()
        store["feat"] = out
    handle = model.layer4.register_forward_hook(fwd_hook)
    x = x.clone().to(device).requires_grad_(True)
    out = model(x)
    probs = torch.softmax(out, dim=1)[0]
    pred = int(out.argmax(1))
    model.zero_grad()
    out[0, pred].backward()
    handle.remove()
    feat = store["feat"]
    weights = feat.grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * feat).sum(dim=1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), pred, probs.detach().cpu().numpy()


def show_cam_grid(paths, titles, suptitle=None):
    # Each image: original (left) and Grad-CAM overlay (right)
    _, tf = get_transforms()
    nrows = int(np.ceil(len(paths) / 2))
    fig, axes = plt.subplots(nrows, 4, figsize=(13, 3.4 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, (p, t) in enumerate(zip(paths, titles)):
        img = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        cam, pred, probs = grad_cam(final_model, tf(Image.open(p).convert("RGB")).unsqueeze(0))
        r, c = divmod(k, 2)
        axes[r, 2 * c].imshow(img)
        axes[r, 2 * c].set_title(t, fontsize=9)
        axes[r, 2 * c + 1].imshow(img)
        axes[r, 2 * c + 1].imshow(cam, cmap="jet", alpha=0.45)
        axes[r, 2 * c + 1].set_title(f"Grad-CAM: {'dog' if pred == 1 else 'cat'} ({probs[pred]:.2f})", fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Test images: no labels, so judge by eye. Put the file names you find WRONG here, e.g. ["12.jpg"]
MANUAL_WRONG_FILES = []

if MANUAL_WRONG_FILES:
    chosen = test_df[test_df.file.isin(MANUAL_WRONG_FILES)]
    suptitle = "Test images judged wrong"
else:
    chosen = pd.concat([test_df.sort_values("confidence").head(4),
                        test_df[test_df.pred == 0].sort_values("confidence", ascending=False).head(2),
                        test_df[test_df.pred == 1].sort_values("confidence", ascending=False).head(2)])
    suptitle = "Test: 4 least confident, then the most confident cats and dogs"
show_cam_grid(chosen.path, [f"{r.file}: {r.label_name} ({r.confidence:.2f})" for r in chosen.itertuples()],
              suptitle=suptitle)

In [ ]:
# Validation errors (labels known), most confident first
wrong_val = val_df[val_df.true != val_df.pred].sort_values("conf", ascending=False).head(8)
names = {0: "cat", 1: "dog"}
print(f"{int((val_df.true != val_df.pred).sum())} validation errors; showing {len(wrong_val)}")
if len(wrong_val):
    show_cam_grid(wrong_val.path,
                  [f"true: {names[r.true]} | pred: {names[r.pred]} ({r.conf:.2f})" for r in wrong_val.itertuples()],
                  suptitle="Most confident validation errors")

## 8. Models and data processing (ablations)

In [ ]:
# Same subset, same lr, one factor changed per row
ABLATIONS = [
    dict(name="ResNet-18 pretrained, fine-tuned, augmentation (reference)"),
    dict(name="... without data augmentation", augment=False),
    dict(name="... backbone frozen (feature extraction only)", freeze_backbone=True),
    dict(name="... smaller images (128 x 128)", img_size=128),
    dict(name="ResNet-18 from scratch (random weights)", pretrained=False, backbone_lr_mult=1.0),
    dict(name="ResNet-50 pretrained, fine-tuned", arch="resnet50"),
]

abl_rows = []
for cfg in ABLATIONS:
    cfg = dict(cfg)
    name = cfg.pop("name")
    r = run_experiment(lr=ABL_LR, epochs=ABL_EPOCHS, **cfg)
    abl_rows.append({"configuration": name, "val_acc": r["best_val_acc"],
                     "train_acc_last": r["history"]["train_acc"][-1],      # measured on augmented images when augmentation is on
                     "train_time_s": round(r["train_time_s"])})
    print(f"{name}: val acc {r['best_val_acc']:.4f}")

In [ ]:
# Conventional classifier: logistic regression on frozen ResNet-18 features
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

@torch.no_grad()
def extract_features(fe_model, loader):
    fe_model.eval()
    feats, labels = [], []
    for x, y in loader:
        feats.append(fe_model(x.to(device)).cpu())
        labels.extend(y.tolist())
    return torch.cat(feats).numpy(), np.array(labels)

set_seed()
tr_l, va_l, _ = make_loaders(TUNE_TRAIN_PER_CLASS, TUNE_VAL_PER_CLASS, augment=False)
fe = build_model("resnet18", pretrained=True)
fe.fc = nn.Identity()
t0 = time.time()
Xtr, ytr = extract_features(fe, tr_l)
Xva, yva = extract_features(fe, va_l)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf.fit(Xtr, ytr)
abl_rows.append({"configuration": "Logistic regression on frozen ResNet-18 features",
                 "val_acc": float(clf.score(Xva, yva)), "train_acc_last": float(clf.score(Xtr, ytr)),
                 "train_time_s": round(time.time() - t0)})

abl_df = pd.DataFrame(abl_rows)
abl_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.55 * len(abl_df) + 1.5))
y_pos = np.arange(len(abl_df))[::-1]
ax.barh(y_pos, abl_df.val_acc, color="#4c78a8")
ax.set_yticks(y_pos); ax.set_yticklabels(abl_df.configuration, fontsize=8)
ax.set_xlim(max(0.0, abl_df.val_acc.min() - 0.05), 1.05)
for yv, v in zip(y_pos, abl_df.val_acc):
    ax.text(v + 0.002, yv, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlabel("validation accuracy"); ax.set_title("Effect of model and data-processing choices")
plt.tight_layout()
plt.show()